# 04 · Model Selection and Development
Compare Logistic Regression, Random Forest and HistGradientBoosting (default settings) on a validation split.

**Reads:** `data/processed/train.parquet`, `model/feature_config.joblib`  
**Writes:** `outputs/model_comparison.csv`

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_recall_curve, roc_curve,
                             confusion_matrix, classification_report, f1_score)
from sklearn.calibration import calibration_curve

from pathlib import Path

# Repo layout (notebooks live in <repo>/notebooks/, everything else is resolved from the repo root):
#   data/raw        <- original Kaggle csv files (flights.csv, airlines.csv, airports.csv)
#   data/processed  <- intermediate files passed from one notebook to the next
#   model/          <- saved models / configs / final bundle
#   outputs/        <- small reports (csv tables)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "model"
OUT_DIR = ROOT / "outputs"
for d in (DATA_RAW, DATA_PROC, MODEL_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42


In [ ]:
cfg = joblib.load(MODEL_DIR / "feature_config.joblib")
FEATURES, TARGET = cfg["features"], cfg["target"]
train = pd.read_parquet(DATA_PROC / "train.parquet")
print("train:", train.shape, "| features:", len(FEATURES))

### Helpers

In [ ]:
def best_threshold(y, p):
    """Probability threshold that maximises F1."""
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return thr[np.argmax(f1[:-1])]

def get_model(name):
    if name == "Logistic Regression":
        return Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=500))])
    if name == "Random Forest":
        return Pipeline([("clf", RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=50,
                                                        n_jobs=-1, random_state=SEED))])
    return Pipeline([("clf", HistGradientBoostingClassifier(random_state=SEED))])

MODELS = ["Logistic Regression", "Random Forest", "HistGradientBoosting"]


### Compare candidate models

In [ ]:
sample = train.sample(min(300_000, len(train)), random_state=SEED)
X_fit, X_val, y_fit, y_val = train_test_split(sample[FEATURES], sample[TARGET], test_size=0.25,
                                              stratify=sample[TARGET], random_state=SEED)
rows, val_proba = [{"model": "Baseline (average)", "roc_auc": 0.5, "pr_auc": y_val.mean(), "f1": np.nan, "fit_sec": 0}], {}
for name in MODELS:
    t0 = time.time()
    model = get_model(name).fit(X_fit, y_fit)
    p = model.predict_proba(X_val)[:, 1]; val_proba[name] = p
    thr = best_threshold(y_val, p)
    rows.append({"model": name, "roc_auc": roc_auc_score(y_val, p), "pr_auc": average_precision_score(y_val, p),
                 "f1": f1_score(y_val, p >= thr), "fit_sec": round(time.time() - t0, 1)})
comparison = pd.DataFrame(rows).set_index("model").round(4)
comparison

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for name, p in val_proba.items():
    fpr, tpr, _ = roc_curve(y_val, p);  ax[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y_val, p):.3f})")
    pr, rc, _ = precision_recall_curve(y_val, p);  ax[1].plot(rc, pr, label=name)
ax[0].plot([0, 1], [0, 1], "k--", lw=.7); ax[0].set(title="ROC curve (validation)", xlabel="False-positive rate", ylabel="True-positive rate"); ax[0].legend()
ax[1].axhline(y_val.mean(), color="k", ls="--", lw=.7); ax[1].set(title="Precision-Recall curve", xlabel="Recall", ylabel="Precision"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
comparison.to_csv(OUT_DIR / "model_comparison.csv")